In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, make_scorer, confusion_matrix

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

import joblib

RANDOM_STATE = 42

In [2]:
df_cluster4 = pd.read_csv("cluster_4.csv")

In [3]:
df_cluster4.describe()

,Index,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,...,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability,cluster
count,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,...,2.313000e+03,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.000000,2313.0,2313.000000,2313.0
mean,2883.313446,0.024211,0.504170,0.560902,0.553719,0.605368,0.605345,0.999027,0.797461,0.809373,...,1.132296e+07,0.623869,0.605366,0.841351,0.282234,0.028320,0.566230,1.0,0.025783,4.0
std,1663.045850,0.153737,0.027483,0.024526,0.026282,0.007949,0.007928,0.000075,0.000100,0.000085,...,2.879249e+08,0.008050,0.007949,0.001264,0.004151,0.017538,0.006942,0.0,0.006047,0.0
min,0.000000,0.000000,0.405011,0.490951,0.460945,0.587332,0.587303,0.998602,0.796861,0.808777,...,1.135300e-04,0.531839,0.587327,0.837671,0.277478,0.000000,0.468982,1.0,0.012028,4.0
25%,1443.000000,0.000000,0.484766,0.544102,0.535093,0.599850,0.599857,0.998984,0.797404,0.809327,...,1.140497e-03,0.623539,0.599851,0.840495,0.279565,0.026817,0.565311,1.0,0.021245,4.0
50%,2884.000000,0.000000,0.501779,0.559256,0.551261,0.603843,0.603871,0.999016,0.797445,0.809359,...,2.563536e-03,0.623770,0.603845,0.841303,0.281025,0.026902,0.565690,1.0,0.025491,4.0
75%,4286.000000,0.000000,0.521572,0.577082,0.569088,0.608952,0.608967,0.999056,0.797503,0.809407,...,6.081374e-03,0.623972,0.608950,0.842198,0.283471,0.027154,0.566630,1.0,0.030053,4.0
max,5805.000000,1.000000,0.609223,0.651603,0.657476,0.657778,0.657778,0.999457,0.798224,0.810030,...,9.650000e+09,0.956387,0.657775,0.847191,0.328568,0.540672,0.735958,1.0,0.044840,4.0


In [4]:
top40 = joblib.load("top_features_for_clustering.joblib")
features_to_use = top40

X_sub = df_cluster4[top40].copy()
y_sub = df_cluster4["Bankrupt?"].copy()

print("Cluster 4 shape:", X_sub.shape)
print("Target Distribution:\n", y_sub.value_counts())

Cluster 4 shape: (2313, 40)
Target Distribution:
 Bankrupt?
0    2257
1      56
Name: count, dtype: int64


In [ ]:
## Defining Base Models

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=RANDOM_STATE
)

knn = KNeighborsClassifier(n_neighbors=5)

base_estimators = [
    ('rf', rf),
    ('gb', gb),
    ('knn', knn)
]

In [ ]:
# Defining Meta Models
meta_model = LogisticRegression(
    penalty="l2",
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Stacking Pipeline
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

# Pipeline: Scale the data first
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('stacking', stacking_clf)
])

In [ ]:
# Model fitting and evaluation
model_pipeline.fit(X_sub, y_sub)

# Predict on the original cluster-4 train rows
y_pred = model_pipeline.predict(X_sub)

cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
TT = tp
TF = fn
N_features = len(features_to_use)
eq1_acc = TT / (TF + TT) if (TF + TT) > 0 else 0

print("-" * 40)
print("RESULTS FOR TABLE 3 (Cluster 4 Model)")
print("-" * 40)
print(f"Confusion Matrix:\n{cm}")
print(f"TT (True Bankrupts Caught): {TT}")
print(f"TF (Bankrupts Missed): {TF}")
print(f"N_features: {N_features}")
print(f"Eq(1) Accuracy: {eq1_acc:.4f}")
print("-" * 40)

----------------------------------------
RESULTS FOR TABLE 3 (Cluster 4 Model)
----------------------------------------
Confusion Matrix:
[[2107  150]
 [   0   56]]
TT (True Bankrupts Caught): 56
TF (Bankrupts Missed): 0
N_features: 40
Eq(1) Accuracy: 1.0000
----------------------------------------


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:165: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:165: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:165: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Libr

In [ ]:
cluster4_package = {
    "cluster_id": 4,
    "features": features_to_use,
    "pipeline": model_pipeline,
    "table3_stats": {"TT": TT, "TF": TF, "Eq1_acc": eq1_acc, "N_features": N_features}
}

joblib.dump(cluster4_package, "cluster4_stacking_B.joblib")
print("Saved cluster4_stacking_B.joblib")

Saved cluster4_stacking_B.joblib
